# CryptoHypothesisLab — 2/2: Istraživanje bez Drive mounta

**Pokreni ISKLJUČIVO u potpuno novoj Colab sesiji** (poslije `Runtime → Disconnect and delete runtime` iz prethodnog notebooka). Ovaj notebook **nema** `drive.mount`, OAuth tokene ni izvornu Drive putanju. Prenosiš samo prethodno preuzeti `HYPOLAB_DEV_*.zip` iz računara.

Paket je samo Discovery 2020–2024: bez Validation/OOS i bez labela. Ovdje izračunati budući ishodi služe **samo za istraživačko otkrivanje**, ne kao nezavisna potvrda edgea. Ne šalji rezultate ili ZIP u javni GitHub/PR/Actions.


In [ ]:
# 1. Sigurnosni preflight PRIJE svakog uploada.
import os
from pathlib import Path
if any(Path(p).exists() for p in ['/content/drive/MyDrive','/content/gdrive/MyDrive','/gdrive/MyDrive']):
    raise RuntimeError('Drive je montiran. Izbrisi runtime i otvori potpuno novu sesiju.')
if any(os.environ.get(k) for k in ('GOOGLE_APPLICATION_CREDENTIALS','GOOGLE_OAUTH_ACCESS_TOKEN','GOOGLE_DRIVE_TOKEN')):
    raise RuntimeError('U okruzenju je Google credential/token; odbijeno.')
print('Analyst preflight PASS: nema prepoznatljivog montiranog Drivea/tokena.')


In [ ]:
# 2. Instaliraj JAVNI kod i odaberi iskljucivo svoj privatni ZIP.
import subprocess,sys
repo=Path('/content/hypolab-research-code')
if repo.exists(): raise RuntimeError('Koristi svjezu sesiju.')
subprocess.run(['git','clone','--depth','1','https://github.com/DuStER1921/CryptoHypothesisLab.git',str(repo)],check=True,stdout=subprocess.DEVNULL)
sys.path.insert(0,str(repo/'src'))
from hypolab.airgap_bundle import inspect_bundle
from hypolab.offline_discovery import describe_forward_return
from google.colab import files
uploaded=files.upload()  # SAMO paket, bez Drive mounta
if len(uploaded)!=1: raise RuntimeError('Dozvoljen je tacno jedan HYPOLAB_DEV ZIP.')
name,payload=next(iter(uploaded.items()))
if not name.startswith('HYPOLAB_DEV_') or not name.endswith('.zip'):
    raise RuntimeError('Pogresan format paketa.')
manifest,rows=inspect_bundle(payload)
print('Development SHA-256 + UTC + C2 status PASS:',manifest['month'],'redova:',len(rows))
print('Izvorni commit:',manifest['producer_code_commit'][:12]);print('OOS nije prisutan.')


In [ ]:
# 3. Prvi opisni pregled, bez statistickih tvrdnji o edgeu.
# Svaka izmjena praga/horizonta je novi Discovery pokusaj; cuvaj SVE nalaze.
import json
from datetime import datetime,timezone
from hashlib import sha256
threshold=0.40
horizon=5
result=describe_forward_return(payload,imbalance_threshold=threshold,horizon_minutes=horizon)
report={'v':1,'kind':'DEVELOPMENT_DESCRIPTIVE_PREVIEW',
        'bundle_sha256':sha256(payload).hexdigest(),
        'generated_utc':datetime.now(timezone.utc).isoformat(),
        'result':result}
print(json.dumps(report,indent=2))


In [ ]:
# 4. Sacuvaj trag eksperimenta ISKLJUCIVO privatno (preuzimanje).
# Ne spremati u javni GitHub, i ne odbacivati neuspjesne rezultate.
from pathlib import Path
p=Path('/content/hypolab-development-preview.json')
if p.exists(): raise RuntimeError('Report vec postoji; ne presnimavati.')
p.write_text(json.dumps(report,sort_keys=True,ensure_ascii=False,indent=2)+'\n',encoding='utf-8')
files.download(str(p))
print('Premjesti privatni izvjestaj u svoj research-results folder ako zelis trajnu evidenciju.')


---
## 5. Kvantitativni screening — 1.440 unaprijed definisanih hipoteza

Ovaj dodatni korak koristi **isti prethodno provjereni ZIP**; ne otvaraj i ne montiraj Google Drive. Pretražujemo smjer C2 neravnoteže, 6 pragova, 6 horizonta, 5 kauzalnih režima i 4 UTC prozora. Plan sadrži SVE hipoteze i moraš ga preuzeti **prije** izračunavanja ishoda. Ovo je Development screening, nije stvarna HFT arbitraža ni validacija.


In [ ]:
# 5. Registruj kompletan plan; sacuvaj datoteku prije prvog masovnog testa.
from hypolab.quant_screen import make_default_plan
from hashlib import sha256
import os
study_plan = make_default_plan(payload)
plan_path = Path('/content/hypolab-quant-plan.json')
with open(plan_path,'xb') as f:
    f.write(study_plan); f.flush(); os.fsync(f.fileno())
assert plan_path.read_bytes()==study_plan
print('PLAN REGISTERED (local, not independent cloud authority)')
print('Plan SHA-256:', sha256(study_plan).hexdigest())
print('Trials:',len(json.loads(study_plan)['trials']))
files.download(str(plan_path))
print('POTVRDI da je plan preuzet prije pokretanja sljedece celije.')


In [ ]:
# 6. Izvrsi SVE planirane triale, bez biranja samo pozitivnih rezultata.
from hypolab.quant_screen import run_registered_screen
quant_report=run_registered_screen(payload,study_plan)
report_path=Path('/content/hypolab-quant-screen.json')
report_raw=(json.dumps(quant_report,ensure_ascii=True,sort_keys=True,allow_nan=False,
                       separators=(',',':'))+'\n').encode('utf-8')
with open(report_path,'xb') as f:
    f.write(report_raw);f.flush();os.fsync(f.fileno())
assert report_path.read_bytes()==report_raw
print('SCREEN COMPLETE (still DEVELOPMENT only)')
print('Trials:',quant_report['total_trials'],'sufficient:',quant_report['sufficient_training_trials'])
print('Plan SHA-256:',quant_report['plan_sha256'])
print('Report SHA-256:',sha256(report_raw).hexdigest())
for trial_id in quant_report['selected_by_training_only'][:5]:
    x=next(t for t in quant_report['all_trials'] if t['trial_id']==trial_id)
    print(trial_id,'maxT p=',x['p_maxT_train'],
          'train net2 bps=',round(x['training']['net_mean_bps']['2'],4),
          'later Dev net2 bps=',round(x['forward_development']['net_mean_bps']['2'],4)
          if x['forward_development']['net_mean_bps']['2'] is not None else None)
files.download(str(report_path))
print('Sacuvaj plan i SVE rezultate privatno; ne salji ih u javni GitHub.')
